# Data Mining Lab — Sreang Florist (Tugas Akhir)
## Analisis Segmentasi Pelanggan RFM (K-Means) & Aturan Asosiasi Produk (FP-Growth)

Notebook ini digunakan untuk mengolah, menganalisis, dan memvalidasi dataset hasil ekspor dari modul Admin sistem **Sreang Florist** (`kmeans_rfm.csv` dan `fp_growth.csv`).

### Tech Stack Lab:
- `pandas`, `numpy`
- `matplotlib`, `seaborn`
- `scikit-learn` (K-Means, Silhouette Score, StandardScaler)
- `mlxtend` (FP-Growth, Association Rules)

In [ ]:
# 1. Install & Import Libraries
!pip install -q mlxtend pandas matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✓ Seluruh library berhasil diimpor.')

---
## BAGIAN 1: SEGMENTASI PELANGGAN RFM DENGAN K-MEANS
Dataset: `kmeans_rfm.csv` (diekspor dari endpoint `/api/export/kmeans`)

In [ ]:
# 2. Load Dataset RFM
# Jika di Google Colab, unggah file kmeans_rfm.csv melalui panel files
try:
    df_rfm = pd.read_csv('kmeans_rfm.csv')
    print(f'Total data pelanggan: {len(df_rfm)}')
    display(df_rfm.head())
except FileNotFoundError:
    print('File kmeans_rfm.csv tidak ditemukan. Harap unggah file terlebih dahulu.')

In [ ]:
# 3. Standarisasi Fitur RFM (Recency, Frequency, Monetary)
features = ['Recency_Days', 'Frequency_Count', 'Monetary_Value']
X = df_rfm[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Evaluasi Jumlah Klaster Optimal (Elbow Method & Silhouette Score)
k_range = range(2, min(7, len(df_rfm)))
wcss = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_range, wcss, 'bo-', label='WCSS (Elbow)')
ax1.set_xlabel('Nilai K (Jumlah Klaster)')
ax1.set_ylabel('Inertia / WCSS', color='b')

ax2 = ax1.twinx()
ax2.plot(k_range, sil_scores, 'ro--', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r')
plt.title('Evaluasi K-Means RFM: Elbow vs Silhouette Score')
plt.show()

In [ ]:
# 5. Pemodelan Final K-Means (K=3)
kmeans_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_rfm['Cluster'] = kmeans_final.fit_predict(X_scaled)

# Profiling Klaster
cluster_profile = df_rfm.groupby('Cluster')[features].mean().reset_index()
cluster_profile['Count'] = df_rfm.groupby('Cluster')['Customer_ID'].count().values

# Urutkan klaster berdasarkan Monetary dan beri label Tier
cluster_profile = cluster_profile.sort_values(by='Monetary_Value', ascending=False).reset_index(drop=True)
tier_labels = ['Gold Tier (VIP)', 'Silver Tier (Loyal)', 'Bronze Tier (Regular)']
cluster_profile['Tier_Status'] = tier_labels

print('=== PROFIL KLASTER K-MEANS RFM ===')
display(cluster_profile)

---
## BAGIAN 2: ATURAN ASOSIASI DENGAN FP-GROWTH
Dataset: `fp_growth.csv` (diekspor dari endpoint `/api/export/fpgrowth`)

In [ ]:
# 6. Load Dataset Transaksional Itemset
try:
    df_fp = pd.read_csv('fp_growth.csv')
    print(f'Total transaksi itemset: {len(df_fp)}')
    display(df_fp.head())
except FileNotFoundError:
    print('File fp_growth.csv tidak ditemukan. Harap unggah file terlebih dahulu.')

In [ ]:
# 7. Transformasi Transaksi ke Format One-Hot Matrix
transactions = [row.split(', ') for row in df_fp['Itemsets'].dropna()]

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# 8. Mining Frequent Itemsets dengan FP-Growth
frequent_patterns = fpgrowth(df_encoded, min_support=0.15, use_colnames=True)
frequent_patterns['length'] = frequent_patterns['itemsets'].apply(lambda x: len(x))
frequent_patterns = frequent_patterns.sort_values(by='support', ascending=False).reset_index(drop=True)

print(f'Total Frequent Itemsets Ditemukan: {len(frequent_patterns)}')
display(frequent_patterns.head(10))

In [ ]:
# 9. Ekstraksi Association Rules (Support, Confidence, Lift)
rules = association_rules(frequent_patterns, metric='confidence', min_threshold=0.5)
rules = rules.sort_values(by='lift', ascending=False).reset_index(drop=True)

print(f'Total Aturan Asosiasi (Confidence >= 50%): {len(rules)}')
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

# Visualisasi Scatter Rules (Support vs Confidence diwarnai Lift)
plt.figure(figsize=(8, 5))
scatter = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='YlOrRd', alpha=0.8, s=60)
plt.colorbar(scatter, label='Lift Ratio')
plt.xlabel('Support')
plt.ylabel('Confidence')
plt.title('Association Rules Sreang Florist (Support vs Confidence vs Lift)')
plt.show()